# Lab 12 Image Retrieval: Task 3 Baseline

Image retrieval focuses on finding one or more images that are the most similar to a query image. This process requires computing the similarity between the query images and all the search candidates.

Image Retrieval can be a supervised task, when there are catalogues that rank a subset of candidate images according to their similarity to some query image. Nevertheless, as we do not possess a ground-truth for the pool tables, we will consider alternative methods to compute the similarity between images and use it to retrieve images.

In [ ]:
import torch
from torchvision.transforms import v2 as transforms
from torch.nn import functional as F
import torchvision
import matplotlib.pyplot as plt
import os, random, re
import pandas as pd
import numpy as np
from PIL import Image

In [ ]:
!unzip images.zip

### Data Preparation

One of the first steps in an image retrieval pipeline is to define the retrieval pool: the set of candidate images among which we want to retrieve the most similar image to a query image.

This set can be the entire training data. Nevertheless, in large datasets, it may be computationally unfeasible to calculate the similarity between the query image and all the data instances. As such, in these cases, we may want to first select a subset of images that can represent our data (e.g., through clustering techniques to find a subset of images that are representative of the entire data).

Since the pool dataset is small, we will consider the entire training data as our retrieval pool.

Let's start by loading the data.

In [ ]:
class PoolDataset:
    def __init__(self, root, partition_file, partition, transform=None):
        self.root = root
        partition_file = pd.read_csv(partition_file)

        idx = np.where(np.asarray(partition_file['partition'].values) == partition)[0]
        self.files = np.asarray(partition_file['image_name'].values)[idx]
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        image = Image.open(os.path.join(self.root, self.files[i]))
        if self.transform:
            image = self.transform(image)
        return image

Define transformations to be applied to the images and data loaders

In [ ]:
# Define transformations to be applied to validation data
val_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToDtype(torch.float32, True),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Define dataloader for the retrieval pool (training data) and query images (test data)
retrieval_pool = PoolDataset('images', 'partition.csv', 'train', val_transform)
query_data = PoolDataset('images', 'partition.csv', 'test', val_transform)

As a use case, let's consider the first test image.

In [ ]:
query_image = query_data[0]
query_image_show = query_image.permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
plt.imshow(query_image_show)
_ = plt.axis("off")

Now that we have both the retrieval pool and query data ready to be used, we can start developing the retrieval methods.

The second step in a retrieval pipeline, after defining the retrieval pool and loading the data, is to define a similarity (or distance) measurement to compare the query image with each of the candidates.

We can compute the similarity between two images on two different levels:
* Image Level Similarity: Measurements computed based on the pixels of the two images.
* Representation-Level Similarity: Measurements computed based on the representations of the two images, obtained through a feature extraction network.

## Retrieval using Image Level Similarity

Let's start by using image-level similarity measurements for retrieving the top-5 most similar images. In this example, we will consider two similarity/distance measurements, available on skimage:
* Euclidean distance/Mean Squared Error: distance measurement (where lower values indicate higher similarity) between the pixels of the two images.
* Structural similarity: similarity measurement (where higher values indicate higher similarity) that considers the structure, luminance and contrast of the image rather than simply comparing the distance between pixels in identical locations of the two images.

In [ ]:
from skimage.metrics import mean_squared_error, structural_similarity

### Mean Squared Error

Calculate the distance between the query and retrieval candidates and select the top-5 images.

In [ ]:
similarities = np.zeros((len(retrieval_pool),))
for i in range(len(retrieval_pool)):
  # calculate the distance between the query image and each candidate
  # torch tensors must be transformed into numpy vectors before computing the mean squared error using skimage
  similarities[i] = # TODO

# select the indexes of the top-5 most similar images
top5_idx = # TODO

Show the top-5 images.

In [ ]:
f, axarr = plt.subplots(2, 3, figsize=(12, 8))
axarr[0][0].imshow(query_image_show)
axarr[0][0].set_title("original")
axarr[0][0].axis("off")
for j in range(1, 6):
  img_show = retrieval_pool[top5_idx[j-1]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
  axarr[j // 3][j % 3].imshow(img_show)
  axarr[j // 3][j % 3].set_title(f"rank = {j}, dist={similarities[top5_idx[j-1]]:.2f}")
  axarr[j // 3][j % 3].axis("off")

Since we do not have a ground-truth to compare the results of retrieval, we need to consider other ways to evaluate the results. Since we want to obtain similar pool tables, whose balls are located in similar positions relative to the table (independently of the view), then, intuitively, the query image should be more similar to itself, after applying some transformation that preserves relative ball position (e.g., translation), than any other image of a pool table.

As a sanity check, let's use the previous similarity/distance measurement to compare the similarity between the query and its most similar image with the similarity between the query and itself, after a small transformation.

Apply a translation of 5 pixels to the query image and compare the distance obtained between the query and its translated version with the distance between the query and the top1 retrieval result.

In [ ]:
# apply translation of 5 pixels to the right to the query image
translate_query_image = # TODO
# calculate the mean squared error between the query and its translated version
sim_trans = # TODO

f, axarr = plt.subplots(1, 3, figsize=(12, 4))
axarr[0].imshow(query_image_show)
axarr[0].set_title("original")
axarr[0].axis("off")

translate_query_image_show = translate_query_image.permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[1].imshow(translate_query_image_show)
axarr[1].set_title(f"translation, dist = {sim_trans:.4f}")
axarr[1].axis("off")

img_show = retrieval_pool[top5_idx[0]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[2].imshow(img_show)
axarr[2].set_title(f"retrieved, dist = {similarities[top5_idx[0]]:.4f}")
axarr[2].axis("off")

What can you conclude from this sanity check?

### Structural Similarity Index Measure

Now, let's repeat the retrieval process but with the structural similarity index measure. This measure provides values in the range [0, 1], where 1 represents the maximum similarity possible between two images.

In [ ]:
similarities = np.zeros((len(retrieval_pool),))
for i in range(len(retrieval_pool)):
  # calculate the similarities between the query image and the candidate using structural similarity 
  similarities[i] = # TODO

# select the indexes of the top-5 most similar images
top5_idx = # TODO

Show the results.

In [ ]:
f, axarr = plt.subplots(2, 3, figsize=(12, 8))
axarr[0][0].imshow(query_image_show)
axarr[0][0].set_title("original")
axarr[0][0].axis("off")
for j in range(1, 6):
  img_show = retrieval_pool[top5_idx[j-1]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
  axarr[j // 3][j % 3].imshow(img_show)
  axarr[j // 3][j % 3].set_title(f"rank = {j}, sim = {similarities[top5_idx[j-1]]:.2f}")
  axarr[j // 3][j % 3].axis("off")

Perform sanity check.

In [ ]:
# apply translation of 5 pixels to the right to the query image
translate_query_image = # TODO
# calculate the structural similarity between the query and its translated version
sim_trans = # TODO

f, axarr = plt.subplots(1, 3, figsize=(12, 4))
axarr[0].imshow(query_image_show)
axarr[0].set_title("original")
axarr[0].axis("off")

translate_query_image_show = translate_query_image.permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[1].imshow(translate_query_image_show)
axarr[1].set_title(f"translation, sim = {sim_trans:.4f}")
axarr[1].axis("off")

img_show = retrieval_pool[top5_idx[0]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[2].imshow(img_show)
axarr[2].set_title(f"retrieved, sim = {similarities[top5_idx[0]]:.4f}")
axarr[2].axis("off")

Notice how all the images retrieved using image-level similarity measurements have the exact same pose. The problem with these metrics is that they only take into consideration pixel-level or structural similarity, while disregarding semantic information. In the image retrieval task, we are more concerned with obtaining pool tables with balls located in similar positions relative to the table, rather than just obtaining tables in similar poses. As such, we need to develop models capable of capturing semantic information.

## Retrieval using Representation-Level Similarity with a Pretrained Model

As a first step towards considering semantic information in the retrieval task, let's leverage the capacity of deep learning models to extract semantically rich features from images. Let's start by using the backbone of a pretrained model (e.g. ResNet) to extract features from images and measure their similarity.

In [ ]:
class FeatureExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # load resnet backbone (or any other backbone of your choosing)
        backbone = # TODO
        # get the feature extractor of the pretrained model (by removing the classification head from the model)
        self.encoder = # TODO
        # freeze the model
        for param in self.encoder.parameters():
            param.requires_grad = False

    def forward(self, x):
        x = self.encoder(x)
        return x

In [ ]:
model = FeatureExtractor()

Now that we have a model that translates images into vectorial representations, we can use it to calculate the distance (or similarity) between images. Like in image-level similarity, we also need to define a similarity/distance measure to compare the vectorial representations. Common measurements include:
* Mean squared error
* Cosine similarity

Note that structural similarity is an image similarity metric and cannot be used with vectorial representations.

In [ ]:
# apply model to obtain the representation of the query image
query_features = # TODO

similarities = np.zeros((len(retrieval_pool),))
for i in range(len(retrieval_pool)):
  # apply model to obtain the representation of the retrieval candidate
  train_features = # TODO
  # compute mean squared error distance between query and candidate representations
  similarities[i] = # TODO

# get indexes of top 5 most similar images
top5_idx = # TODO

Show the results.

In [ ]:
f, axarr = plt.subplots(2, 3, figsize=(12, 8))
axarr[0][0].imshow(query_image_show)
axarr[0][0].set_title("original")
axarr[0][0].axis("off")
for j in range(1, 6):
  img_show = retrieval_pool[top5_idx[j-1]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
  axarr[j // 3][j % 3].imshow(img_show)
  axarr[j // 3][j % 3].set_title(f"rank = {j}, dist = {similarities[top5_idx[j-1]]:.4f}")
  axarr[j // 3][j % 3].axis("off")

Perform sanity check.

In [ ]:
# apply translation of 5 pixels to the query image
translate_query_image = # TODO
# get the augmented image's features
translate_query_features = # TODO
# calculate the mean squared error between the features of the query image and its augmented version
sim_trans = # TODO

f, axarr = plt.subplots(1, 3, figsize=(12, 4))
axarr[0].imshow(query_image_show)
axarr[0].set_title("original")
axarr[0].axis("off")

translate_query_image_show = translate_query_image.permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[1].imshow(translate_query_image_show)
axarr[1].set_title(f"translation, dist = {sim_trans:.4f}")
axarr[1].axis("off")

img_show = retrieval_pool[top5_idx[0]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[2].imshow(img_show)
axarr[2].set_title(f"retrieved, dist = {similarities[top5_idx[0]]:.4f}")
axarr[2].axis("off")

The pretrained model was not optimized on the pool dataset and, thus, has a limited capacity to capture semantically meaningful features. As such, this model may not be ideal for the retrieval task, and may need to be optimized for this task. As there are no ground-truth labels for retrieval, we can consider two main methods for optimizing the feature extractor:
 * Direct optimization: Directly optimizing the feature extractor by computing some loss function directly on the features extracted by the feature extractor - we need to decide which loss function and associated training technique can be used to optimize the network for retrieval
 * Proxy Task: Training the feature extractor as part of a network developed for a task other than retrieval, but that requires extracting rich semantic features relative to the pool game from the image - we need to decide which task can be used for this


## Optimizing the Pretrained Model for Retrieval through Direct Optimization



In the retrieval task, an image should always be the most similar to itself, independently of any transformation that does not change the relative position of the balls. Through the sanity checks, we saw that previous methods did not consider the query image to be similar to itself after a very small translation. To fix this behaviour, we can directly optimize the feature extractor to minimize the distance to itself.

Nevertheless, simply minimizing the distance between augmented versions of the same image is not enough as it can lead to the network simply compacting the latent space and minimizing the distance between all data samples. As such, we also need to maximize the distance between different images.

This training process with a loss function where we minimize the distance between specific data samples while maximizing it for other samples is called Contrastive Learning. In contrastive learning, we need positive pairs with samples whose distance we want to minimize, and negative pairs whose distance we want to maximize. To account for data imbalance, let's build a data loader that can be used to obtain three samples at once: the original image, its augmented version and a randomly selected image.

In [ ]:
class TripletPoolDataset:
    def __init__(self, root, partition_file, partition, transform=None, val_transform=None):
        self.root = root
        partition_file = pd.read_csv(partition_file)

        idx = np.where(np.asarray(partition_file['partition'].values) == partition)[0]
        self.files = np.asarray(partition_file['image_name'].values)[idx]
        self.transform = transform
        self.val_transform = val_transform
        self.partition = partition

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        image = Image.open(os.path.join(self.root, self.files[i]))

        if self.partition == "train":
            # select a random image of the data (except the current image at index i)
            candidate_idx = np.concatenate((np.arange(len(self.files)-1)[:i], np.arange(len(self.files)-1)[i+1:]))
            neg_idx = random.choice(candidate_idx)
        else:
            # in the validation set, we need to make sure that the negative pair is always the same 
            # to better monitor the performance of the model
            neg_idx = i+1
            if neg_idx >= len(self.files):
                neg_idx = 0

        neg_image = Image.open(os.path.join(self.root, self.files[neg_idx]))

        # apply transformations to the query, negative (randomly selected) sample and positive (augmented) sample
        image_query = # TODO
        image_pos = # TODO
        image_neg = # TODO

        return image_query, image_pos, image_neg

Define datasets and data loaders.

In [ ]:
aug_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.Resize((256, 256)),
    # TODO - Add data augmentation functions here (e.g., affine transformations and colour alterations)
    transforms.CenterCrop((224, 224)),
    transforms.ToDtype(torch.float32, True),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = TripletPoolDataset('images', 'partition.csv', 'train', aug_transform, val_transform)
val_dataset = TripletPoolDataset('images', 'partition.csv', 'valid', aug_transform, val_transform)

batch_size = 8
num_workers = 1
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size, shuffle=True, num_workers=num_workers)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size, num_workers=num_workers)

Define the Feature Extractor model.

In [ ]:
class FeatureExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # load resnet backbone (or any other backbone of your choosing)
        backbone = # TODO
        # get the feature extractor of the pretrained model (by removing the classification head from the model)
        self.encoder = # TODO
        # this time we want to train the encoder, so we do not freeze it

    def forward(self, x):
        x = self.encoder(x)
        return x

In [ ]:
# device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)

# create model
model = FeatureExtractor()
model = model.to(device)

In [ ]:
# Define optimizer
optimizer = torch.optim.AdamW(model.parameters())
epochs = 20

# Define loss
loss_fn = torch.nn.MSELoss(reduction="none")

Define one epoch of the training process. Here, we need to define the contrastive loss function where we maximize the distance between the query image and a randomly selected image of the data (negative sample) and minimize the distance between the query and its augmented version (positive sample).

Note that, when maximizing the distance without any restrictions, the network may lead to infinitely large distances, leading to a sparse latent space. To avoid this, we can restrict the maximum distance between two different samples by defining a margin value such that any value above that margin would lead to the exact same loss (i.e., clip the distance so that it does not go beyond the margin when computing the loss).

In [ ]:
def one_epoch(model, optimizer, dataloader, is_training):
  model.train() if is_training else model.eval()
  avg_loss = 0
  for i, (image_query, image_pos, image_neg) in enumerate(dataloader):
    image_query = image_query.to(device)
    image_pos = image_pos.to(device)
    image_neg = image_neg.to(device)
    # extract representations of the query, positive and negative samples using the feature extractor
    rep_query = # TODO
    rep_pos = # TODO
    rep_neg = # TODO

    # implement here the loss term to minimize the distance between the query and positive sample
    loss = # TODO
    # loss term used to maximize the distance between the query and its augmented version
    m = 1.
    loss += torch.mean(torch.maximum(m - loss_fn(rep_query, rep_neg), torch.zeros((len(image_query))).to(device)))
    
    if is_training:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    avg_loss += loss.item() / len(dataloader)
  return avg_loss

Train the model (note that in contrastive learning, we usually need to train the network for a large number of epochs).

In [ ]:
train_history = {'loss': [], 'metric': []}
val_history = {'loss': [], 'metric': []}
for epoch in range(epochs):
  # compute train
  avg_loss = one_epoch(model, optimizer, train_dataloader, True)
  train_history['loss'].append(avg_loss)
  print(f'Epoch {epoch+1:2d}/{epochs} - Train loss: {avg_loss}')
  # compute validation statistics
  avg_loss = one_epoch(model, optimizer, val_dataloader, False)
  val_history['loss'].append(avg_loss)
  print(f'Epoch {epoch+1:2d}/{epochs} - Val   loss: {avg_loss}')

Get the top-5 most similar images.

In [ ]:
# apply model to obtain the representation of the query image
query_features = # TODO

similarities = np.zeros((len(retrieval_pool),))
for i in range(len(retrieval_pool)):
  # apply model to obtain the representation of the retrieval candidate
  train_features = # TODO
  # compute mean squared error distance between query and candidate representations
  similarities[i] = # TODO

# get indexes of top 5 most similar images
top5_idx = # TODO

Show the results.

In [ ]:
f, axarr = plt.subplots(2, 3, figsize=(12, 8))
axarr[0][0].imshow(query_image_show)
axarr[0][0].set_title("original")
axarr[0][0].axis("off")
for j in range(1, 6):
  img_show = retrieval_pool[top5_idx[j-1]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
  axarr[j // 3][j % 3].imshow(img_show)
  axarr[j // 3][j % 3].set_title(f"rank = {j}, dist = {similarities[top5_idx[j-1]]:.4f}")
  axarr[j // 3][j % 3].axis("off")

Perform sanity check.

In [ ]:
# apply translation of 5 pixels to the query image
translate_query_image = # TODO
# get the augmented image's features
translate_query_features = # TODO
# calculate the mean squared error between the features of the query image and its augmented version
sim_trans = # TODO

f, axarr = plt.subplots(1, 3, figsize=(12, 4))
axarr[0].imshow(query_image_show)
axarr[0].set_title("original")
axarr[0].axis("off")

translate_query_image_show = translate_query_image.permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[1].imshow(translate_query_image_show)
axarr[1].set_title(f"translation, dist = {sim_trans:.4f}")
axarr[1].axis("off")

img_show = retrieval_pool[top5_idx[0]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[2].imshow(img_show)
axarr[2].set_title(f"retrieved, dist = {similarities[top5_idx[0]]:.4f}")
axarr[2].axis("off")

In this example with contrastive learning, we are maximizing the distance between random pairs of images of the data in the same manner, independently of how similar these images may be. However, in the dataset, there are some cases of different images that may be more similar, and others more different (e.g., an image with only 2 balls is likely more different to an image with 10 balls than another image with 10 balls). Can you think of a way to improve the contrastive learning loss function to account for potential differences in the similarity between different images?

**Additional Suggestion**: Notice that, in the dataset, there are images with the letters a, t and f next to the image number. These images typically represent different views of the same pool table. These images should be considered as similar as possible by the retrieval model. How can you change the code (training or data loading process) to account for these images?  

## Optimizing the Pretrained Model for Retrieval through Proxy Task

As a proxy task, let's train the feature extractor as part of an autoencoder trained for image reconstruction.

Get the training and validation dataset and dataloader.

In [ ]:
aug_transform = transforms.Compose([
    transforms.ToImage(),
    transforms.Resize((256, 256)),
    # TODO - define here some data augmentations for training the image reconstruction model 
    # (these are not strictly needed but can help avoid overfitting)
    transforms.CenterCrop((224, 224)),
    transforms.ToDtype(torch.float32, True),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = PoolDataset('images', 'partition.csv', 'train', aug_transform)
val_dataset = PoolDataset('images', 'partition.csv', 'valid', val_transform)

batch_size = 8
num_workers = 2

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size, shuffle=True, num_workers=num_workers)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size, num_workers=num_workers)

Implement the autoencoder model.

Note: Notebook 9 (Semantic Segmentation and Object Detection) contained an example/exercise with an autoencoder trained for segmentation, you can reuse its architecture for image reconstruction.

In [ ]:
class MyAutoencoder(torch.nn.Module):
    def __init__(self, out_channels):
        super().__init__()
        # load resnet backbone (or any other backbone of your choosing)
        backbone = # TODO
        # get the feature extractor of the pretrained model (by removing the classification head and global average pooling layer from the model)
        self.encoder = # TODO
        # add a decoder to the model
        self.decoder = # TODO
        # define the last layer of the model
        self.out = # TODO

    def forward(self, x):
         # TODO
        return x

In [ ]:
# device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)

model = MyAutoencoder(3)
model = model.to(device)

In [ ]:
# Define optimizer
optimizer = torch.optim.AdamW(model.parameters())
epochs = 10

# Define image reconstruction loss
loss_fn = # TODO

Implement one epoch of the training process.

In [ ]:
def one_epoch(model, optimizer, dataloader, is_training):
  model.train() if is_training else model.eval()
  avg_loss = 0
  for i, images in enumerate(dataloader):
    images = images.to(device)
    reconstruction = # TODO
    loss = # TODO
    if is_training:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    avg_loss += loss.item() / len(dataloader)
  return avg_loss

Train the model.

In [ ]:
train_history = {'loss': [], 'metric': []}
val_history = {'loss': [], 'metric': []}
for epoch in range(epochs):
  # compute train
  avg_loss = one_epoch(model, optimizer, train_dataloader, True)
  train_history['loss'].append(avg_loss)
  print(f'Epoch {epoch+1:2d}/{epochs} - Train loss: {avg_loss}')
  # compute validation statistics
  avg_loss = one_epoch(model, optimizer, val_dataloader, False)
  val_history['loss'].append(avg_loss)
  print(f'Epoch {epoch+1:2d}/{epochs} - Val   loss: {avg_loss}')

Get the top-5 most similar images.

In [ ]:
# apply encoder of the model to obtain the representation of the query image
query_features = # TODO

similarities = np.zeros((len(retrieval_pool),))
for i in range(len(retrieval_pool)):
  # apply encoder of the model to obtain the representation of the retrieval candidate
  train_features = # TODO
  # compute mean squared error distance between query and candidate representations
  similarities[i] = # TODO

# get indexes of top 5 most similar images
top5_idx = # TODO

Show the results.

In [ ]:
f, axarr = plt.subplots(2, 3, figsize=(12, 8))
axarr[0][0].imshow(query_image_show)
axarr[0][0].set_title("original")
axarr[0][0].axis("off")
for j in range(1, 6):
  img_show = retrieval_pool[top5_idx[j-1]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
  axarr[j // 3][j % 3].imshow(img_show)
  axarr[j // 3][j % 3].set_title(f"rank = {j}, dist = {similarities[top5_idx[j-1]]:.4f}")
  axarr[j // 3][j % 3].axis("off")

Perform sanity check.

In [ ]:
# apply translation of 5 pixels to the query image
translate_query_image = # TODO
# get the augmented image's features by applying the model's encoder
translate_query_features = # TODO
# calculate the mean squared error between the features of the query image and its augmented version
sim_trans = # TODO

f, axarr = plt.subplots(1, 3, figsize=(12, 4))
axarr[0].imshow(query_image_show)
axarr[0].set_title("original")
axarr[0].axis("off")

translate_query_image_show = translate_query_image.permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[1].imshow(translate_query_image_show)
axarr[1].set_title(f"translation, dist = {sim_trans:.4f}")
axarr[1].axis("off")

img_show = retrieval_pool[top5_idx[0]].permute(1, 2, 0)*torch.tensor([[[0.229, 0.224, 0.225]]]) + torch.tensor([[[0.485, 0.456, 0.406]]])
axarr[2].imshow(img_show)
axarr[2].set_title(f"retrieved, dist = {similarities[top5_idx[0]]:.4f}")
axarr[2].axis("off")

Is image reconstruction a good proxy task to optimize the feature extractor for image retrieval?

Which task would be better?